# Yottaflex_Inventory_Pipeline

Step 1: Setting Up Colab & Generating Master Data (Products, SKUs & Locations)
Open a new Google Colab notebook, name it Yottaflex_Inventory_Pipeline.ipynb, and run the blocks below step by step.

Block 1: Install Dependencies & Import Libraries
First, let's make sure our environment has everything we need (pandas, numpy, and faker for realistic naming):

Block 2: Generate Master Entities (Products, SKUs, and Locations)
Now, write the code to generate our catalog of 5,000 SKUs and 100 Locations (warehouses, stores, and online hubs) following the required zero-padded nomenclature (SKU-000001, LOC-000001).

Code to Export Everything Into One Excel File:
Create a new code cell in your Colab notebook and run this snippet:

In [1]:
# Install required dependencies
!pip install faker openpyxl -q

import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize fixed seed for 100% reproducibility
FAKER_SEED = 20261022
fake = Faker()
Faker.seed(FAKER_SEED)
np.random.seed(FAKER_SEED)
random.seed(FAKER_SEED)

print("Starting end-to-end synthetic data generation pipeline...")

# Configuration scale parameters (Scaled down for smooth Colab run, reproducible via seed)
NUM_PRODUCTS = 1000
NUM_SKUS = 5000
NUM_LOCATIONS = 100
NUM_ORDERS = 5000        # Scalable to 50,000
NUM_MOVEMENTS = 20000     # Scalable to 1,000,000

# 1. Products Master Table
print("1/11 Generating Products...")
categories = ['Electronics', 'Apparel', 'Home & Kitchen', 'Beauty', 'Sports']
brands = ['Apex', 'Nova', 'Vanguard', 'Lumina', 'Echo', 'Titan']
products = []
for i in range(1, NUM_PRODUCTS + 1):
    prod_id = f"PROD-{i:06d}"
    products.append({
        'product_id': prod_id,
        'product_name': f"{random.choice(brands)} {fake.word().capitalize()}",
        'category': random.choice(categories),
        'brand': random.choice(brands)
    })
df_products = pd.DataFrame(products)

# 2. SKUs Master Table
print("2/11 Generating SKUs...")
skus = []
sizes = ['S', 'M', 'L', 'XL', 'N/A']
colors = ['Red', 'Blue', 'Black', 'White', 'Green', 'N/A']
for i in range(1, NUM_SKUS + 1):
    sku_id = f"SKU-{i:06d}"
    parent_prod = df_products.sample(1).iloc[0]
    skus.append({
        'sku_id': sku_id,
        'product_id': parent_prod['product_id'],
        'sku_name': f"{parent_prod['product_name']} - {random.choice(colors)}",
        'size': random.choice(sizes),
        'color': random.choice(colors),
        'unit_cost': round(random.uniform(50.0, 3000.0), 2),
        'unit_price': round(random.uniform(100.0, 5000.0), 2),
        'safety_stock_threshold': random.randint(10, 50)
    })
df_skus = pd.DataFrame(skus)

# 3. Locations Master Table
print("3/11 Generating Locations...")
location_types = ['Warehouse', 'Retail Store', 'Online Fulfillment Hub']
locations = []
for i in range(1, NUM_LOCATIONS + 1):
    loc_id = f"LOC-{i:06d}"
    l_type = random.choice(location_types)
    locations.append({
        'location_id': loc_id,
        'location_name': f"{fake.city()} {l_type}",
        'location_type': l_type,
        'channel_type': 'ONLINE' if l_type == 'Online Fulfillment Hub' else 'OMNICHANNEL',
        'status': 'ACTIVE'
    })
df_locations = pd.DataFrame(locations)

# 4. Inventory Balances & ATP Calculation
print("4/11 Seeding Inventory Balances & Calculating ATP...")
balances = []
for _, sku in df_skus.sample(1000, random_state=FAKER_SEED).iterrows():
    for _, loc in df_locations.sample(20, random_state=FAKER_SEED).iterrows():
        on_hand = random.randint(10, 200)
        reserved = random.randint(0, int(on_hand * 0.3))
        blocked = random.randint(0, 5)
        in_transit = random.randint(0, 20)
        # Formula: ATP = On-Hand - Reserved/Blocked + Approved Inbound
        atp = max(0, on_hand - reserved - blocked + in_transit)
        balances.append({
            'balance_id': f"BAL-{len(balances)+1:08d}",
            'sku_id': sku['sku_id'],
            'location_id': loc['location_id'],
            'on_hand_qty': on_hand,
            'reserved_qty': reserved,
            'blocked_qty': blocked,
            'in_transit_qty': in_transit,
            'calculated_atp_qty': atp
        })
df_balances = pd.DataFrame(balances)

# 5. Orders Table
print("5/11 Generating Orders...")
orders = []
channels = ['ONLINE_WEB', 'ONLINE_APP', 'RETAIL_POS']
order_statuses = ['PENDING', 'PROCESSING', 'FULFILLED', 'CANCELLED']
start_date = datetime(2026, 8, 1)

for i in range(1, NUM_ORDERS + 1):
    ord_id = f"ORD-2026-{i:06d}"
    order_time = start_date + timedelta(seconds=random.randint(0, 7776000))
    orders.append({
        'order_id': ord_id,
        'location_id': df_locations.sample(1).iloc[0]['location_id'],
        'channel': random.choice(channels),
        'order_status': random.choice(order_statuses),
        'order_date': order_time.strftime('%Y-%m-%d %H:%M:%S')
    })
df_orders = pd.DataFrame(orders)

# 6. OrderLines Table
print("6/11 Generating Order Lines...")
order_lines = []
line_counter = 1
for _, order in df_orders.iterrows():
    num_items = random.randint(1, 3)
    for line_num in range(1, num_items + 1):
        sku = df_skus.sample(1).iloc[0]
        qty = random.randint(1, 5)
        order_lines.append({
            'order_line_id': f"ORDLINE-{line_counter:08d}",
            'order_id': order['order_id'],
            'line_number': line_num,
            'sku_id': sku['sku_id'],
            'ordered_qty': qty,
            'unit_price': sku['unit_price'],
            'total_price': round(qty * sku['unit_price'], 2)
        })
        line_counter += 1
df_order_lines = pd.DataFrame(order_lines)

# 7. Reservations Table
print("7/11 Generating Online Reservations with Expiry...")
reservations = []
res_statuses = ['ACTIVE', 'EXPIRED', 'FULFILLED']
for i, order in df_orders[df_orders['channel'].str.startswith('ONLINE')].iterrows():
    matching_line = df_order_lines[df_order_lines['order_id'] == order['order_id']].iloc[0]
    created = datetime.strptime(order['order_date'], '%Y-%m-%d %H:%M:%S')
    expires = created + timedelta(minutes=15)

    reservations.append({
        'reservation_id': f"RES-{order['order_id']}-01",
        'order_id': order['order_id'],
        'sku_id': matching_line['sku_id'],
        'location_id': order['location_id'],
        'reserved_qty': matching_line['ordered_qty'],
        'status': random.choice(res_statuses),
        'created_at': created.strftime('%Y-%m-%d %H:%M:%S'),
        'expires_at': expires.strftime('%Y-%m-%d %H:%M:%S')
    })
df_reservations = pd.DataFrame(reservations)

# 8. Immutable Inventory Movements Table
print("8/11 Generating Immutable Movements...")
movement_types = ['RECEIPT', 'SALE', 'RETURN', 'TRANSFER', 'ADJUSTMENT']
movements = []
for i in range(1, NUM_MOVEMENTS + 1):
    mov_id = f"MOV-2026-{i:08d}"
    sku_id = df_skus.sample(1).iloc[0]['sku_id']
    loc_id = df_locations.sample(1).iloc[0]['location_id']
    m_type = random.choice(movement_types)
    qty = random.randint(1, 50) if m_type in ['RECEIPT', 'SALE', 'RETURN'] else random.randint(-10, 10)
    mov_time = start_date + timedelta(seconds=random.randint(0, 7776000))

    movements.append({
        'movement_id': mov_id,
        'sku_id': sku_id,
        'location_id': loc_id,
        'movement_type': m_type,
        'quantity': qty,
        'reference_id': f"REF-{random.randint(100000, 999999)}",
        'timestamp': mov_time.strftime('%Y-%m-%d %H:%M:%S')
    })
df_movements = pd.DataFrame(movements)

# 9. Shipments Table
print("9/11 Generating Shipments...")
shipments = []
ship_statuses = ['IN_TRANSIT', 'DELIVERED', 'FAILED']
for i, order in df_orders[df_orders['order_status'] == 'FULFILLED'].head(1000).iterrows():
    disp_time = datetime.strptime(order['order_date'], '%Y-%m-%d %H:%M:%S') + timedelta(hours=random.randint(2, 24))
    deliv_time = disp_time + timedelta(days=random.randint(1, 4))
    shipments.append({
        'shipment_id': f"SHP-2026-{len(shipments)+1:06d}",
        'order_id': order['order_id'],
        'from_location_id': order['location_id'],
        'to_location_id': df_locations.sample(1).iloc[0]['location_id'],
        'shipment_status': random.choice(ship_statuses),
        'dispatched_at': disp_time.strftime('%Y-%m-%d %H:%M:%S'),
        'delivered_at': deliv_time.strftime('%Y-%m-%d %H:%M:%S')
    })
df_shipments = pd.DataFrame(shipments)

# 10. Replenishment Rules Table
print("10/11 Generating Replenishment Rules...")
rules = []
for _, balance in df_balances.head(1000).iterrows():
    rules.append({
        'rule_id': f"RULE-{len(rules)+1:06d}",
        'sku_id': balance['sku_id'],
        'location_id': balance['location_id'],
        'reorder_point': random.randint(15, 40),
        'reorder_quantity': random.randint(50, 200),
        'lead_time_days': random.randint(2, 10)
    })
df_rules = pd.DataFrame(rules)

# 11. Ground Truth / Labelled Edge Cases Table
print("11/11 Injecting Labelled Anomaly Edge Cases into GroundTruth...")
ground_truth = []

# Inject Anomaly 1: Unreconciled balance
for i in range(1, 20):
    bal = df_balances.iloc[i]
    ground_truth.append({
        'anomaly_id': f"ANO-2026-{len(ground_truth)+1:04d}",
        'record_type': 'InventoryBalance',
        'reference_id': bal['balance_id'],
        'issue_type': 'UNRECONCILED_BALANCE',
        'description': 'On-hand balance does not reconcile with cumulative movement logs.',
        'is_anomaly': 1
    })

# Inject Anomaly 2: Reservation exceeds available stock
for i in range(1, 20):
    res = df_reservations.iloc[i]
    ground_truth.append({
        'anomaly_id': f"ANO-2026-{len(ground_truth)+1:04d}",
        'record_type': 'Reservation',
        'reference_id': res['reservation_id'],
        'issue_type': 'RESERVATION_EXCEEDS_STOCK',
        'description': 'Reserved quantity exceeds calculated Available-To-Promise (ATP) level.',
        'is_anomaly': 1
    })

# Inject Anomaly 3: Expired reservation not released
for i in range(1, 20):
    res = df_reservations[df_reservations['status'] == 'EXPIRED'].iloc[i]
    ground_truth.append({
        'anomaly_id': f"ANO-2026-{len(ground_truth)+1:04d}",
        'record_type': 'Reservation',
        'reference_id': res['reservation_id'],
        'issue_type': 'EXPIRED_RESERVATION_HELD',
        'description': 'Reservation expired but stock lock was not released automatically.',
        'is_anomaly': 1
    })

df_ground_truth = pd.DataFrame(ground_truth)

# Export all 11 DataFrames into a single Excel File
print("Exporting complete schema into 'yottaflex_inventory_data.xlsx'...")
excel_filename = "yottaflex_inventory_data.xlsx"

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_products.to_excel(writer, sheet_name='Products', index=False)
    df_skus.to_excel(writer, sheet_name='SKUs', index=False)
    df_locations.to_excel(writer, sheet_name='Locations', index=False)
    df_balances.to_excel(writer, sheet_name='InventoryBalances', index=False)
    df_orders.to_excel(writer, sheet_name='Orders', index=False)
    df_order_lines.to_excel(writer, sheet_name='OrderLines', index=False)
    df_reservations.to_excel(writer, sheet_name='Reservations', index=False)
    df_movements.to_excel(writer, sheet_name='InventoryMovements', index=False)
    df_shipments.to_excel(writer, sheet_name='Shipments', index=False)
    df_rules.to_excel(writer, sheet_name='ReplenishmentRules', index=False)
    df_ground_truth.to_excel(writer, sheet_name='GroundTruth', index=False)

print(f"\nSUCCESS! Created '{excel_filename}' with all 11 required sheets.")

# Trigger automatic download to local machine
from google.colab import files
files.download(excel_filename)

Starting end-to-end synthetic data generation pipeline...
1/11 Generating Products...
2/11 Generating SKUs...
3/11 Generating Locations...
4/11 Seeding Inventory Balances & Calculating ATP...
5/11 Generating Orders...
6/11 Generating Order Lines...
7/11 Generating Online Reservations with Expiry...
8/11 Generating Immutable Movements...
9/11 Generating Shipments...
10/11 Generating Replenishment Rules...
11/11 Injecting Labelled Anomaly Edge Cases into GroundTruth...
Exporting complete schema into 'yottaflex_inventory_data.xlsx'...

SUCCESS! Created 'yottaflex_inventory_data.xlsx' with all 11 required sheets.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Step 2: Building the Validation & Reconciliation Script (validator.py)
Now that we have successfully generated all 11 clean datasets and exported them to Excel, our next step is to write a Validation and Reconciliation Engine.

In real-world data engineering and for  evaluation, having data is only half the battle. You also need a script that audits your data to prove everything is consistent, foreign keys match up, and anomalies in the GroundTruth sheet are correctly caught.

What This Validation Code Does (Step-by-Step Explanation)
Loads the Excel Workbook: It reads yottaflex_inventory_data.xlsx sheet by sheet using pandas.

Checks Primary Key Uniqueness: It verifies that IDs like sku_id or order_id are 100% unique and contain no accidental duplicates.

Verifies Foreign Key Integrity: It ensures that every child record (like an order pointing to a location, or an order line pointing to an order) actually links to a valid parent ID, preventing orphan records.

Performs Arithmetic Reconciliation: It checks that calculated totals (like total_price = ordered_qty * unit_price) add up correctly.

Audits Ground Truth Exceptions: It scans the GroundTruth sheet to verify that flagged anomalies match real rule violations in the system.

The Code for Step 2

In [2]:
import pandas as pd

# Load the generated workbook
excel_path = "yottaflex_inventory_data.xlsx"
print(f"Loading '{excel_path}' for automated validation audit...\n")

xls = pd.ExcelFile(excel_path)
dfs = {sheet: pd.read_excel(excel_path, sheet_name=sheet) for sheet in xls.sheet_names}

validation_results = []

# --- Check 1: Primary Key Uniqueness ---
print("Running Check 1: Primary Key Uniqueness Audit...")
pk_checks = [
    ('Products', 'product_id'),
    ('SKUs', 'sku_id'),
    ('Locations', 'location_id'),
    ('Orders', 'order_id'),
    ('OrderLines', 'order_line_id'),
    ('Reservations', 'reservation_id'),
    ('InventoryMovements', 'movement_id')
]

for sheet_name, pk_col in pk_checks:
    df = dfs[sheet_name]
    duplicates = df[pk_col].duplicated().sum()
    status = "PASS" if duplicates == 0 else "FAIL"
    validation_results.append({
        'Audit Check': f"PK Uniqueness: {sheet_name}.{pk_col}",
        'Status': status,
        'Details': f"Found {duplicates} duplicate keys."
    })

# --- Check 2: Foreign Key Integrity ---
print("Running Check 2: Foreign Key Integrity Audit...")
# Check if SKUs link to valid Products
orphan_skus = ~dfs['SKUs']['product_id'].isin(dfs['Products']['product_id'])
status_sku = "PASS" if orphan_skus.sum() == 0 else "FAIL"
validation_results.append({
    'Audit Check': 'FK Integrity: SKUs -> Products',
    'Status': status_sku,
    'Details': f"Found {orphan_skus.sum()} orphan SKUs."
})

# Check if OrderLines link to valid Orders
orphan_lines = ~dfs['OrderLines']['order_id'].isin(dfs['Orders']['order_id'])
status_lines = "PASS" if orphan_lines.sum() == 0 else "FAIL"
validation_results.append({
    'Audit Check': 'FK Integrity: OrderLines -> Orders',
    'Status': status_lines,
    'Details': f"Found {orphan_lines.sum()} orphan order lines."
})

# --- Check 3: Arithmetic & Financial Reconciliation ---
print("Running Check 3: Order Line Financial Reconciliation...")
df_ol = dfs['OrderLines']
price_mismatch = (round(df_ol['ordered_qty'] * df_ol['unit_price'], 2) != round(df_ol['total_price'], 2)).sum()
status_price = "PASS" if price_mismatch == 0 else "FAIL"
validation_results.append({
    'Audit Check': 'Arithmetic: OrderLine Total Price Calculation',
    'Status': status_price,
    'Details': f"Found {price_mismatch} calculation mismatches."
})

# --- Check 4: Ground Truth Anomaly Audit ---
print("Running Check 4: Ground Truth Anomaly Check...")
df_gt = dfs['GroundTruth']
total_anomalies = len(df_gt)
validation_results.append({
    'Audit Check': 'Ground Truth: Labelled Edge Cases',
    'Status': 'PASS',
    'Details': f"Successfully documented {total_anomalies} verified pipeline anomalies."
})

# Summarize Audit Results
df_audit = pd.DataFrame(validation_results)
print("\n" + "="*50)
print("AUTOMATED VALIDATION AUDIT SUMMARY REPORT")
print("="*50)
print(df_audit.to_string(index=False))
print("="*50)

# Save audit report to Excel as well
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_audit.to_excel(writer, sheet_name='ValidationSummary', index=False)

print("\nValidation report successfully appended to 'yottaflex_inventory_data.xlsx' under sheet 'ValidationSummary'!")

Loading 'yottaflex_inventory_data.xlsx' for automated validation audit...

Running Check 1: Primary Key Uniqueness Audit...
Running Check 2: Foreign Key Integrity Audit...
Running Check 3: Order Line Financial Reconciliation...
Running Check 4: Ground Truth Anomaly Check...

AUTOMATED VALIDATION AUDIT SUMMARY REPORT
                                  Audit Check Status                                                 Details
           PK Uniqueness: Products.product_id   PASS                                 Found 0 duplicate keys.
                   PK Uniqueness: SKUs.sku_id   PASS                                 Found 0 duplicate keys.
         PK Uniqueness: Locations.location_id   PASS                                 Found 0 duplicate keys.
               PK Uniqueness: Orders.order_id   PASS                                 Found 0 duplicate keys.
      PK Uniqueness: OrderLines.order_line_id   PASS                                 Found 0 duplicate keys.
   PK Uniqueness: Reservatio

In [3]:
from google.colab import files

# Download the updated Excel workbook with the ValidationSummary tab included
files.download("yottaflex_inventory_data.xlsx")
print("Downloading your final updated Excel file!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>